# EV Remaining Range Prediction
### Linear Regression vs Decision Tree vs Random Forest

**Updated for the current VoltTrack dataset** (`EV_Dataset_2025_15000_Cars_Final_180K_Realistic.xlsx`):
15,000 vehicles &times; 12 monthly readings each = ~180,000 rows.

Change from the previous version of this notebook: the road-type column is
now called **`route_type`** (not `road_type`), and it has **three**
categories instead of two — `City`, `Highway`, and `Mixed`. All references
have been updated to match.


In [ ]:
# ============================================================
# EV REMAINING RANGE PREDICTION
# Linear Regression vs Decision Tree vs Random Forest
# ============================================================

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

## 1. Load the Dataset

In [ ]:
# Update this path if your file is saved somewhere else
file_path = r"C:\Users\dhine\Downloads\EV_Dataset_2025_15000_Cars_Final_180K_Realistic.xlsx"

df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)   # expect roughly (180000, 63)
df.head()

In [ ]:
import os
print(os.path.exists(file_path))

In [ ]:
print(df.columns.tolist())

## 2. Build the Target Variable

`remaining_range_km` is derived from the vehicle's rated `range_km` and its
current `soc_percent` (battery charge level) at that reading.

In [ ]:
df["remaining_range_km"] = (
    df["range_km"] * df["soc_percent"] / 100
)

print(
    df[["range_km", "soc_percent", "remaining_range_km"]].head(10)
)

## 3. Select Features

Same feature set as before, with `road_type` corrected to **`route_type`**
to match this dataset's real column name (values: `City`, `Highway`,
`Mixed`).

In [ ]:
features = [
    "vehicle_model",
    "motor_power_kw",
    "soc_percent",
    "route_type",
    "torque",
    "length_mm",
    "width_mm",
    "height_mm",
    "wheel_base_mm",
    "battery_capacity_kwh",
    "weight_kg",
]

X = df[features].copy()
print(X.columns.tolist())

In [ ]:
categorical_features = [
    "vehicle_model",
    "route_type",
]

numeric_features = [
    "motor_power_kw",
    "soc_percent",
    "torque",
    "length_mm",
    "width_mm",
    "height_mm",
    "wheel_base_mm",
    "battery_capacity_kwh",
    "weight_kg",
]

In [ ]:
y = df["remaining_range_km"]

## 4. One-Hot Encode Categorical Features

`route_type` will now expand into **three** dummy columns
(`route_type_City`, `route_type_Highway`, `route_type_Mixed`) instead of the
two the old dataset produced.

In [ ]:
X = pd.get_dummies(
    X,
    columns=["vehicle_model", "route_type"],
    dtype=int,
)

print(X.head())
print(X.shape)

In [ ]:
print(X.shape)
print(y.shape)
print(X.dtypes)

## 5. Add Realistic Noise to the Target

The raw formula `range_km * soc_percent / 100` is a perfect deterministic
relationship, which would let the models "cheat" and score near-perfect R².
Adding a small amount of random noise (15% of the value) makes the
prediction task more realistic, the same approach used in the original
notebook.

In [ ]:
np.random.seed(42)

noise = np.random.normal(
    loc=0,
    scale=0.15 * df["remaining_range_km"],
)

df["remaining_range_km_noisy"] = (
    df["remaining_range_km"] + noise
)

print(
    df[["remaining_range_km", "remaining_range_km_noisy"]].head(10)
)

In [ ]:
y = df["remaining_range_km_noisy"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 6. Linear Regression

**Note:** with ~180,000 rows now (12x more than before), this and the
Random Forest step below will take noticeably longer to run than they did
on the old 15,000-row dataset — that's expected, just let them finish.

In [ ]:
reg = LinearRegression()
reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)
print(y_pred[:10])

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Linear Regression")
print("-----------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

## 7. Random Forest Regression

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Regression")
print("------------------------")
print("MAE  :", rf_mae)
print("RMSE :", rf_rmse)
print("R2   :", rf_r2)

## 8. Decision Tree Regression

In [ ]:
dt = DecisionTreeRegressor(
    random_state=42,
    max_depth=12,
)

dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [ ]:
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
dt_r2 = r2_score(y_test, dt_pred)

print("Decision Tree Regression")
print("------------------------")
print("MAE  :", dt_mae)
print("RMSE :", dt_rmse)
print("R2   :", dt_r2)

## 9. Compare the Three Models

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
    ],
    "MAE": [mae, dt_mae, rf_mae],
    "RMSE": [rmse, dt_rmse, rf_rmse],
    "R2 Score": [r2, dt_r2, rf_r2],
})

print(results)

## 10. Save the Best Model

Saves whichever model scored highest on R² to a `.pkl` file using
`joblib` (already imported at the top), so it can be loaded later —
for example, from your VoltTrack backend to serve real range predictions.

In [ ]:
models = {
    "Linear Regression": (reg, r2),
    "Decision Tree": (dt, dt_r2),
    "Random Forest": (rf, rf_r2),
}

best_name, (best_model, best_r2) = max(models.items(), key=lambda item: item[1][1])
print(f"Best model: {best_name} (R2 = {best_r2:.4f})")

output_path = Path("best_ev_range_model.pkl")
joblib.dump(best_model, output_path)
print(f"Saved to: {output_path.resolve()}")

# Also save the exact column order X was trained on, so new data can be
                           # aligned correctly before calling model.predict() later.
joblib.dump(list(X.columns), Path("model_feature_columns.pkl"))